In [22]:
import sys
import os
import torch
import torchvision
import ultralytics

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import cv2
from PIL import Image

from pathlib import Path
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import shutil

In [23]:
print("Python:", sys.version)
print("Python executable:", sys.executable)

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Python executable: c:\Users\ezycloudx-admin\Desktop\ALL-Det_raven172_processing\.venv\Scripts\python.exe
PyTorch: 2.13.0+cu130
TorchVision: 0.28.0+cu130
Ultralytics: 8.4.174
OpenCV: 5.0.0


In [24]:
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute Capability:", torch.cuda.get_device_capability(0))

    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory: {props.total_memory / 1024**3:.2f} GB")

CUDA available: True
PyTorch CUDA: 13.0
GPU: NVIDIA GeForce RTX 5060 Ti
Compute Capability: (12, 0)
GPU Memory: 15.93 GB


In [25]:
device = torch.device("cuda:0")

a = torch.randn(2048, 2048, device=device)
b = torch.randn(2048, 2048, device=device)

c = torch.matmul(a, b)

torch.cuda.synchronize()

print("Device:", c.device)
print("Result shape:", c.shape)

Device: cuda:0
Result shape: torch.Size([2048, 2048])


In [26]:
ROOT = Path(r"C:\Users\ezycloudx-admin\Desktop\LVHH\ALL_IDB1\ALL_IDB1")
OUT = ROOT.with_name("ALL_IDB1_splitted")

if OUT.exists():
    raise FileExistsError("The output directory already exists. Please rename OUT.")

images = sorted((ROOT / "images").glob("*.jpg"))

y = [
    int(bool(
        (ROOT / "labels" / f"{img.stem}.txt")
        .read_text(encoding="utf-8-sig").strip()
    ))
    for img in images
]

train_images, val_images = train_test_split(
    images,
    test_size=round(len(images) * 0.15),
    random_state=0,
    stratify=y,
)

# Copy to a new directory, preserving the original data.
for split, items in {"train": train_images, "val": val_images}.items():
    (OUT / "images" / split).mkdir(parents=True)
    (OUT / "labels" / split).mkdir(parents=True)

    for img in items:
        label = ROOT / "labels" / f"{img.stem}.txt"
        shutil.copy2(img, OUT / "images" / split / img.name)
        shutil.copy2(label, OUT / "labels" / split / label.name)

    print(f"{split}: {len(items)} images")

(OUT / "data.yaml").write_text(
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: blast\n",
    encoding="utf-8",
)

DATA_YAML = str((OUT / "data.yaml").resolve())
print("Dataset saved at:", OUT)
print("data.yaml:", DATA_YAML)

train: 91 images
val: 16 images
Dataset saved at: C:\Users\ezycloudx-admin\Desktop\LVHH\ALL_IDB1\ALL_IDB1_splitted
data.yaml: C:\Users\ezycloudx-admin\Desktop\LVHH\ALL_IDB1\ALL_IDB1_splitted\data.yaml


In [ ]:
DATA = Path(
    r"C:\Users\ezycloudx-admin\Desktop\LVHH\ALL_IDB1\ALL_IDB1_splitted\data.yaml"
)

RESULTS_ROOT = Path(
    r"C:\Users\ezycloudx-admin\Desktop\ALL-Det_raven172_processing\results\RES001_EXP001-results"
)

EXPERIMENT_DIR = RESULTS_ROOT / "YOLOv5nu-results"
RUN_NAME = "YOLOv5nu-results"
MODEL_DIR = EXPERIMENT_DIR / RUN_NAME
WEIGHTS = "yolov5nu.pt"

from ultralytics import YOLO

# Load pretrained model
model = YOLO(WEIGHTS)

# Display model architecture
model.info()

# Train model
results = model.train(
    data=str(DATA),
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    workers=0,
    project=str(EXPERIMENT_DIR),
    name=RUN_NAME,
    seed=42,
    plots=True
)

print(f"Training results: {model.trainer.save_dir}")

YOLOv5n summary: 153 layers, 2,654,816 parameters, 0 gradients, 7.8 GFLOPs
Ultralytics 8.4.174  Python-3.12.10 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ezycloudx-admin\Desktop\LVHH\ALL_IDB1\ALL_IDB1_splitted\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mo

In [ ]:
DATA = Path(
    r"C:\Users\ezycloudx-admin\Desktop\LVHH\ALL_IDB1\ALL_IDB1_splitted\data.yaml"
)

RESULTS_ROOT = Path(
    r"C:\Users\ezycloudx-admin\Desktop\ALL-Det_raven172_processing\results\RES001_EXP001-results"
)

EXPERIMENT_DIR = RESULTS_ROOT / "YOLOv8-results"
RUN_NAME = "YOLOv8-result"
MODEL_DIR = EXPERIMENT_DIR / RUN_NAME
WEIGHTS = "yolov8n.pt"

from ultralytics import YOLO

# Load pretrained model
model = YOLO(WEIGHTS)

# Display model architecture
model.info()

# Train model
results = model.train(
    data=str(DATA),
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    workers=0,
    project=str(EXPERIMENT_DIR),
    name=RUN_NAME,
    seed=42,
    plots=True
)

print(f"Training results: {model.trainer.save_dir}")

YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs
Ultralytics 8.4.174  Python-3.12.10 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ezycloudx-admin\Desktop\LVHH\ALL_IDB1\ALL_IDB1_splitted\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mo

In [32]:
DATA = Path(
    r"C:\Users\ezycloudx-admin\Desktop\LVHH\ALL_IDB1\ALL_IDB1_splitted\data.yaml"
)

RESULTS_ROOT = Path(
    r"C:\Users\ezycloudx-admin\Desktop\ALL-Det_raven172_processing\results\RES001_EXP001-results"
)

EXPERIMENT_DIR = RESULTS_ROOT / "YOLO11-results"
RUN_NAME = "YOLO11-result"
MODEL_DIR = EXPERIMENT_DIR / RUN_NAME
WEIGHTS = "yolo11n.pt"

from ultralytics import YOLO

# Load pretrained model
model = YOLO(WEIGHTS)

# Display model architecture
model.info()

# Train model
results = model.train(
    data=str(DATA),
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    workers=0,
    project=str(EXPERIMENT_DIR),
    name=RUN_NAME,
    seed=42,
    plots=True
)

print(f"Training results: {model.trainer.save_dir}")

YOLO11n summary: 181 layers, 2,624,080 parameters, 0 gradients, 6.7 GFLOPs
Ultralytics 8.4.174  Python-3.12.10 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ezycloudx-admin\Desktop\LVHH\ALL_IDB1\ALL_IDB1_splitted\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mo